# Train PARSeq in Jupyter

Select a Base, Tiny or Fast Tiny profile. Inspect the images, labels, batch tensors and editable PyTorch loop below. Every Fast Tiny phase uses the same training-only holdout. After the final phase, open the inference notebook or inspect the final validation strings here.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "parseq_ocr").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "parseq_ocr").is_dir():
    raise RuntimeError("Open this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))
from parseq_ocr.config import load_config
# Dummy presets: configs/dummy_base.yaml, dummy_tiny.yaml, dummy_fast_tiny.yaml.
# Real-data presets: configs/alpr_base.yaml, alpr_tiny.yaml, alpr_fast_tiny.yaml.
# PARSEQ_CONFIG overrides this default when it is set.
CONFIG_PATH = os.environ.get("PARSEQ_CONFIG", "configs/dummy_tiny.yaml")
config = load_config(CONFIG_PATH)
print("Config:", CONFIG_PATH)
print("Model:", config["model"], "input:", config["image_size"])
print("Artifacts:", config["output_dir"])

# Edit in this cell for a separate experiment:
# config["output_dir"] = ROOT / "artifacts/my_parseq_run"
# config["batch_size"] = 4
# config["learning_rate"] = 0.0001
# Edit in this cell for a separate experiment:
# config["output_dir"] = ROOT / "artifacts/my_parseq_run"
# config["batch_size"] = 4
# config["learning_rate"] = 0.0001

## Inspect the dataset and a batch

In [ ]:
from collections import defaultdict
from PIL import Image, ImageEnhance
from torch.utils.data import Dataset, DataLoader
"""Train-only epoch selection followed by an optional full-training refit."""
from pathlib import Path
import contextlib
import json
import math
import random
import time
import numpy as np
import torch
from parseq_ocr.config import serializable
from parseq_ocr.data import scan_pairs, split_holdout, validate_labels, normalize
from parseq_ocr.models import create_system
from parseq_ocr.metrics import score_pairs, selection_score, require_exact_selection, EXACT_POLICY



class CropDataset(Dataset):
    def __init__(self, records, image_size=(64, 256), augment=False):
        self.records, self.image_size, self.augment = records, image_size, augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(row["path"]) as im:
            image = im.convert("RGB")
        if self.augment:
            if random.random() < .5:
                image = ImageEnhance.Brightness(image).enhance(random.uniform(.7, 1.3))
            if random.random() < .5:
                image = ImageEnhance.Contrast(image).enhance(random.uniform(.7, 1.3))
            if random.random() < .25:
                image = image.rotate(random.uniform(-3, 3), Image.Resampling.BILINEAR,
                                     fillcolor=(127, 127, 127))
        h, w = self.image_size
        image = image.resize((w, h), Image.Resampling.BICUBIC)
        pixels = np.asarray(image, dtype=np.float32).transpose(2, 0, 1).copy()/127.5-1
        return torch.from_numpy(pixels), row["label"], row["image"]


def make_loader(records, config, augment=False):
    return DataLoader(CropDataset(records, config["image_size"], augment),
                      batch_size=config["batch_size"], shuffle=augment,
                      num_workers=config["num_workers"], pin_memory=True,
                      persistent_workers=config["num_workers"] > 0)



records = scan_pairs(config["dataset_root"] / "train", config["normalization"])
fit, holdout = split_holdout(records, config["holdout_fraction"], config["split_seed"])
sample = fit[0]
with Image.open(sample["path"]) as image:
    display(image.copy())
pixels, texts, names = next(iter(make_loader(fit, config)))
print("Label:", repr(sample["label"]), "Batch shape:", tuple(pixels.shape), texts[:4])

## Inspect evaluation and edit the PyTorch training loop

The optimizer, loss selection, gradient clipping, checkpoint selection, and resume state are visible below. The model package provides the PARSeq architecture and loss primitives.

In [ ]:
def autocast(config):
    if config["precision"] == "fp32":
        return contextlib.nullcontext()
    if config["precision"] != "bf16" or config["device"] != "cuda":
        raise ValueError("Use fp32 for CPU or bf16 on a CUDA device supporting BF16")
    if not torch.cuda.is_bf16_supported():
        raise ValueError("This GPU does not support BF16; set precision: fp32")
    return torch.autocast("cuda", dtype=torch.bfloat16)


@torch.inference_mode()
def evaluate_records(system, records, config, output=None):
    system.eval()
    labels, predictions, rows = [], [], []
    for pixels, texts, names in make_loader(records, config):
        with autocast(config):
            logits = system(pixels.to(config["device"]))
        decoded = system.tokenizer.decode(logits.float().softmax(-1))[0]
        for name, gt, pred in zip(names, texts, decoded):
            labels.append(gt)
            predictions.append(pred)
            rows.append({"image": name, "gt": gt, "prediction": pred})
    summary = score_pairs(labels, predictions)
    if output:
        output = Path(output)
        output.mkdir(parents=True, exist_ok=True)
        (output/"summary.json").write_text(json.dumps(summary, indent=2))
        (output/"predictions.jsonl").write_text("".join(json.dumps(r)+"\n" for r in rows))
    return summary


def _seed(config):
    random.seed(config["seed"])
    np.random.seed(config["seed"])
    torch.manual_seed(config["seed"])
    torch.set_num_threads(4)


def _fit(records, held, config, folder, epochs, resume):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    last = folder / "last.pt"
    resume_state = None
    if last.exists():
        if not resume:
            raise FileExistsError(f"Existing training state: {last}; use resume or a new output_dir")
        resume_state = torch.load(last, map_location="cpu", weights_only=False)
        require_exact_selection(resume_state, str(last))
    _seed(config)
    system = create_system(config).to(config["device"])
    validate_labels(records+held, system.tokenizer, config.get("max_label_length", 25))
    system.rng = np.random.default_rng(config["seed"])
    # The reference objective is unchanged; standalone loops do not have a Lightning Trainer.
    system.log = lambda *args, **kwargs: None
    optimizer = torch.optim.AdamW(system.parameters(), lr=config["learning_rate"],
                                 weight_decay=config["weight_decay"])
    training = make_loader(records, config, augment=True)
    total_steps = epochs*len(training)
    start_epoch, steps, selected_epoch, best = 1, 0, 0, (-1., -float("inf"))
    last = folder/"last.pt"
    if resume_state is not None:
        state = resume_state
        system.model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        start_epoch, steps = state["epoch"]+1, state["steps"]
        selected_epoch, best = state["selected_epoch"], tuple(state["best"])
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        torch.set_rng_state(state["torch_rng"].cpu())
        system.rng.bit_generator.state = state["permutation_rng"]
    history_path = folder/"history.jsonl"
    started = time.perf_counter()
    for epoch in range(start_epoch, epochs+1):
        system.train()
        total_loss = 0.
        for index, (pixels, labels, _) in enumerate(training):
            scale = (min(1., (steps+1)/config.get("warmup_steps", 100))*(.1+.9*.5*(1+math.cos(math.pi*steps/total_steps)))
                     if config.get("lr_schedule", "cosine") == "cosine" else 1.)
            for group in optimizer.param_groups:
                group["lr"] = config["learning_rate"]*scale
            optimizer.zero_grad(set_to_none=True)
            if config.get("synthetic_replay_ratio", 0):
                from parseq_ocr.parallel import synthesize
                replay, replay_labels = synthesize(max(1, int(len(labels)*config["synthetic_replay_ratio"])),
                                                   *config["image_size"], config.get("max_label_length", 12))
                pixels = torch.cat([pixels, replay])
                labels = list(labels)+replay_labels
            with autocast(config):
                if config.get("training_objective") == "parallel":
                    from parseq_ocr.parallel import parallel_loss
                    loss = parallel_loss(system, pixels.to(config["device"]), list(labels))
                else:
                    loss = system.training_step((pixels.to(config["device"]), list(labels)), index)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite training loss")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(system.parameters(), 1., error_if_nonfinite=True)
            optimizer.step()
            steps += 1
            total_loss += loss.item()
        row = {"epoch": epoch, "loss": total_loss/len(training),
               "seconds_this_process": time.perf_counter()-started}
        if held:
            row["holdout"] = evaluate_records(system, held, config)
            score = selection_score(row["holdout"])
        else:
            score = (epoch, 0.)
        if score > best:
            best, selected_epoch = score, epoch
            torch.save(system.model.state_dict(), folder/("best.pt" if held else "model.pt"))
        with history_path.open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(row)+"\n")
        torch.save({"selection_policy": EXACT_POLICY, "model": system.model.state_dict(), "optimizer": optimizer.state_dict(),
                    "epoch": epoch, "steps": steps, "selected_epoch": selected_epoch, "best": best,
                    "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
                    "torch_rng": torch.get_rng_state(), "permutation_rng": system.rng.bit_generator.state}, last)
        print(json.dumps(row), flush=True)
    summary = {"selection_policy": EXACT_POLICY, "epochs": epochs, "selected_epoch": selected_epoch,
               "selection_score": list(best) if held else None,
               "training_images": len(records), "holdout_images": len(held),
               "parameters": sum(p.numel() for p in system.parameters())}
    (folder/"summary.json").write_text(json.dumps(summary, indent=2))
    return summary

## Review the Fast Tiny stages

This cell defines the warm-start, parallel adaptation, and smaller-input replay schedule. Base and standard Tiny use one training schedule.

In [ ]:
from parseq_ocr.metrics import require_exact_selection, EXACT_POLICY
"""Leakage-free warm start, parallel adaptation and smaller-input replay stages."""
import json
import hashlib
from PIL import Image
import torch
from parseq_ocr.config import serializable
from parseq_ocr.data import scan_pairs, split_holdout


def _input_fingerprint(records, training, held, replay_ratio):
    rows = []
    fit_ids = {r["image"] for r in training}
    held_ids = {r["image"] for r in held}
    for row in records:
        with Image.open(row["path"]) as image:
            rgb = image.convert("RGB")
            rows.append(dict(image=row["image"], label=row["label"],
                             size=list(rgb.size), rgb_sha256=hashlib.sha256(rgb.tobytes()).hexdigest(),
                             membership="fit" if row["image"] in fit_ids else "held"))
    if fit_ids & held_ids or fit_ids | held_ids != {r["image"] for r in records}:
        raise ValueError("Invalid training/holdout membership")
    return dict(version=1, rows=rows, replay_ratio=replay_ratio,
                fit_order=[r["image"] for r in training],
                held_order=[r["image"] for r in held])


def train_fast_tiny(config, resume=True):
    if config["device"] == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable; choose CPU with precision: fp32 for functional checks")
    if config["epochs"] < 1 or config.get("synthetic_replay_ratio", .25) < 0:
        raise ValueError("epochs must be positive and replay ratio nonnegative")
    output = config["output_dir"]
    output.mkdir(parents=True, exist_ok=True)
    snapshot = serializable(config)
    path = output/"config.json"
    if path.exists() and json.loads(path.read_text()) != snapshot:
        raise ValueError("Config differs from existing run; choose a new output_dir")
    records = scan_pairs(config["dataset_root"]/"train", config["normalization"])
    training, held = split_holdout(records, config["holdout_fraction"], config["split_seed"])
    fingerprint = _input_fingerprint(records, training, held,
                                     config.get("synthetic_replay_ratio", .25))
    fingerprint_path = output/"input_fingerprint.json"
    if fingerprint_path.exists():
        if json.loads(fingerprint_path.read_text()) != fingerprint:
            raise ValueError("Input content or split changed; choose a new output_dir")
    elif any(output.rglob("*.pt")) or any(output.rglob("summary.json")):
        raise ValueError("Legacy run lacks an input fingerprint; choose a new output_dir")
    # Validate identity before overwriting any persisted configuration or split.
    fingerprint_path.write_text(json.dumps(fingerprint, indent=2))
    path.write_text(json.dumps(snapshot, indent=2))
    (output/"split.json").write_text(json.dumps({"train": [r["image"] for r in training],
                                                "holdout": [r["image"] for r in held]}, indent=2))
    max_length = config.get("max_label_length", 12)
    if any(len(r["label"]) > max_length for r in records):
        raise ValueError(f"Fast Tiny supports at most {max_length} characters; labels are never truncated")
    stages = [
        ("warm_start", dict(image_size=[64,256], autoregressive=True, refinements=1,
                            training_objective="permutation", synthetic_replay_ratio=0,
                            learning_rate=config.get("warm_start_learning_rate", 1e-4),
                            lr_schedule="cosine"), config.get("warm_start_epochs", 20)),
        ("parallel_adaptation", dict(image_size=[64,256], autoregressive=False, refinements=2,
                                     training_objective="parallel", synthetic_replay_ratio=0,
                                     learning_rate=config.get("adaptation_learning_rate", 3e-5),
                                     lr_schedule="constant"), config.get("adaptation_epochs", 5)),
        ("final", dict(image_size=config["image_size"], autoregressive=False, refinements=2,
                       training_objective="parallel",
                       synthetic_replay_ratio=config.get("synthetic_replay_ratio", .25),
                       lr_schedule="constant"), config["epochs"]),
    ]
    summaries, previous, previous_size = [], None, None
    for name, overrides, epochs in stages:
        folder = output if name == "final" else output/name
        cfg = dict(config, **overrides)
        cfg.update(recipe="standard", max_label_length=max_length, epochs=epochs, output_dir=folder)
        selection_cfg = dict(cfg)
        if previous:
            selection_cfg.update(initial_checkpoint=previous/"selection/best.pt",
                                 initial_image_size=previous_size)
        summary = folder/"selection/summary.json"
        selected = json.loads(summary.read_text()) if summary.exists() else _fit(
            training, held, selection_cfg, folder/"selection", epochs, resume)
        require_exact_selection(selected, str(summary))
        full_cfg = dict(cfg)
        if previous and config["refit_all_training"]:
            full_cfg.update(initial_checkpoint=previous/"full_fit/model.pt",
                            initial_image_size=previous_size)
        if (folder/"full_fit/summary.json").exists():
            require_exact_selection(json.loads((folder/"full_fit/summary.json").read_text()), str(folder/"full_fit/summary.json"))
        if config["refit_all_training"] and not (folder/"full_fit/summary.json").exists():
            _fit(records, [], full_cfg, folder/"full_fit", selected["selected_epoch"], resume)
        folder.mkdir(parents=True, exist_ok=True)
        (folder/"stage_config.json").write_text(json.dumps({"selection": serializable(selection_cfg),
                                                             "full_fit": serializable(full_cfg)}, indent=2))
        summaries.append(dict(stage=name, **selected))
        previous, previous_size = folder, overrides["image_size"]
    result = {"selection_policy": EXACT_POLICY, "recipe": "fast_tiny", "stages": summaries,
              "selection_protocol": "Every stage uses the same training-only label-disjoint holdout. "
                                    "Selection starts from preceding selection/best.pt; refit starts "
                                    "from preceding full_fit/model.pt when refit_all_training is enabled. Validation is never read.",
              "synthetic_replay": "Random uppercase/digits only; ratio .25 adds 1 synthetic per 4 real images (20% effective batch). "
                                  "Fonts vary by platform; exact historical accuracy is not guaranteed."}
    (output/"recipe_summary.json").write_text(json.dumps(result, indent=2))
    return result

## Train and inspect validation predictions

In [ ]:
def train(config, resume=True):
    if config.get("normalization", "none") != "none":
        raise ValueError("Literal training labels require normalization: none")
    if config.get("recipe") == "fast_tiny":
        return train_fast_tiny(config, resume)
    if config["device"] == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable; use a GPU or device: cpu with precision: fp32")
    if config["epochs"] < 1:
        raise ValueError("epochs must be positive")
    output = Path(config["output_dir"])
    output.mkdir(parents=True, exist_ok=True)
    snapshot = serializable(config)
    if (output/"config.json").exists() and json.loads((output/"config.json").read_text()) != snapshot:
        raise ValueError("Config differs from the existing run. Choose a new output_dir.")
    (output/"config.json").write_text(json.dumps(snapshot, indent=2))
    records = scan_pairs(config["dataset_root"]/"train", config["normalization"])
    training, held = split_holdout(records, config["holdout_fraction"], config["split_seed"])
    (output/"split.json").write_text(json.dumps({"train": [r["image"] for r in training],
                                                "holdout": [r["image"] for r in held]}, indent=2))
    summary = output/"selection/summary.json"
    selection = json.loads(summary.read_text()) if summary.exists() else _fit(
        training, held, config, output/"selection", config["epochs"], resume)
    require_exact_selection(selection, str(summary))
    if config["refit_all_training"]:
        summary = output/"full_fit/summary.json"
        if summary.exists():
            require_exact_selection(json.loads(summary.read_text()), str(summary))
        if not summary.exists():
            _fit(records, [], config, output/"full_fit", selection["selected_epoch"], resume)
    return selection

selection = train(config, resume=True)
checkpoint = config["output_dir"] / ("full_fit/model.pt" if config["refit_all_training"] else "selection/best.pt")
print("Checkpoint:", checkpoint)
if (config["dataset_root"] / "val").is_dir():
    system = create_system(config, checkpoint).to(config["device"])
    validation = scan_pairs(config["dataset_root"] / "val", config["normalization"])
    print(evaluate_records(system, validation, config, config["output_dir"] / "evaluation_native"))
    predictions = config["output_dir"] / "evaluation_native/predictions.jsonl"
    display([json.loads(line) for line in predictions.read_text().splitlines()[:10]])